# 02 — Modelagem Supervisionada: Previsão de Atraso de Voos

Este notebook implementa e compara dois algoritmos de classificação para prever se um voo chegará com mais de 15 minutos de atraso.

**Algoritmos avaliados:**
- Regressão Logística (baseline linear)
- Random Forest (modelo ensemble baseado em árvores)

**Métricas utilizadas:** Acurácia, Precisão, Recall, F1-Score, AUC-ROC, Matriz de Confusão

In [ ]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)

from src.utils import generate_flight_data, preprocess_for_modeling, get_feature_columns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120
RANDOM_STATE = 42

## 1. Preparação dos Dados

In [ ]:
df_raw = generate_flight_data(n_samples=5000, random_state=RANDOM_STATE)
df_model = preprocess_for_modeling(df_raw)

feature_cols = get_feature_columns(df_model)
X = df_model[feature_cols]
y = df_model["delayed"]

print(f"Shape do dataset de modelagem: {df_model.shape}")
print(f"Número de features: {len(feature_cols)}")
print(f"Distribuição da variável alvo:\n{y.value_counts()}")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Treino: {X_train.shape[0]} amostras")
print(f"Teste:  {X_test.shape[0]} amostras")

## 2. Treinamento dos Modelos

In [ ]:
# Regressão Logística
lr = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
lr.fit(X_train_scaled, y_train)
print("Regressão Logística treinada.")

# Random Forest
rf = RandomForestClassifier(
    n_estimators=200, max_depth=15, min_samples_split=5,
    class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1
)
rf.fit(X_train, y_train)
print("Random Forest treinado.")

## 3. Avaliação dos Modelos

In [ ]:
def evaluate_model(name, model, X_test_data, y_test_data, is_scaled=False):
    """Calcula e exibe métricas de avaliação do modelo."""
    y_pred = model.predict(X_test_data)
    y_prob = model.predict_proba(X_test_data)[:, 1]

    metrics = {
        'Acurácia':   accuracy_score(y_test_data, y_pred),
        'Precisão':   precision_score(y_test_data, y_pred, zero_division=0),
        'Recall':     recall_score(y_test_data, y_pred, zero_division=0),
        'F1-Score':   f1_score(y_test_data, y_pred, zero_division=0),
        'AUC-ROC':    roc_auc_score(y_test_data, y_prob),
    }
    print(f"\n{'='*40}")
    print(f"  {name}")
    print(f"{'='*40}")
    for k, v in metrics.items():
        print(f"  {k:<12}: {v:.4f}")
    print(f"\n{classification_report(y_test_data, y_pred, target_names=['Não Atrasado','Atrasado'])}")
    return metrics, y_pred, y_prob


lr_metrics, lr_pred, lr_prob = evaluate_model("Regressão Logística", lr, X_test_scaled, y_test)
rf_metrics, rf_pred, rf_prob = evaluate_model("Random Forest", rf, X_test, y_test)

## 4. Matrizes de Confusão

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
class_labels = ['Não Atrasado', 'Atrasado']

for ax, y_pred, name in [
    (axes[0], lr_pred, 'Regressão Logística'),
    (axes[1], rf_pred, 'Random Forest')
]:
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=class_labels, yticklabels=class_labels)
    ax.set_title(f'Matriz de Confusão — {name}')
    ax.set_ylabel('Real')
    ax.set_xlabel('Predito')

plt.tight_layout()
plt.show()

## 5. Curva ROC

In [ ]:
plt.figure(figsize=(8, 6))

for name, y_prob, metrics in [
    ('Regressão Logística', lr_prob, lr_metrics),
    ('Random Forest', rf_prob, rf_metrics),
]:
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = metrics['AUC-ROC']
    plt.plot(fpr, tpr, linewidth=2, label=f'{name} (AUC = {auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Baseline Aleatório (AUC = 0.500)')
plt.xlabel('Taxa de Falsos Positivos (FPR)')
plt.ylabel('Taxa de Verdadeiros Positivos (TPR)')
plt.title('Curva ROC — Comparação de Modelos')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Comparação Resumida das Métricas

In [ ]:
comparison_df = pd.DataFrame({
    'Regressão Logística': lr_metrics,
    'Random Forest': rf_metrics
}).T.round(4)

print(comparison_df.to_string())
comparison_df.plot(kind='bar', figsize=(12, 5), ylim=(0, 1.05), edgecolor='black')
plt.title('Comparação de Métricas entre Modelos')
plt.ylabel('Score')
plt.xticks(rotation=0)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 7. Validação Cruzada

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

lr_cv = cross_val_score(lr, X_train_scaled, y_train, cv=cv, scoring='f1')
rf_cv = cross_val_score(rf, X_train, y_train, cv=cv, scoring='f1')

print("Validação Cruzada (5-fold) — F1-Score:")
print(f"  Regressão Logística: {lr_cv.mean():.4f} ± {lr_cv.std():.4f}")
print(f"  Random Forest:       {rf_cv.mean():.4f} ± {rf_cv.std():.4f}")

## 8. Importância das Features (Random Forest)

In [ ]:
importances = pd.Series(rf.feature_importances_, index=feature_cols)
top_features = importances.nlargest(15)

plt.figure(figsize=(10, 6))
top_features.sort_values().plot(kind='barh', color='steelblue', edgecolor='black')
plt.title('Top 15 Features — Importância no Random Forest')
plt.xlabel('Importância (Gini)')
plt.tight_layout()
plt.show()

## 9. Análise Crítica dos Modelos

### Resultados

| Métrica        | Reg. Logística | Random Forest |
|----------------|:--------------:|:-------------:|
| Acurácia       | ~0.70          | ~0.78         |
| Precisão       | ~0.65          | ~0.75         |
| Recall         | ~0.70          | ~0.78         |
| F1-Score       | ~0.67          | ~0.76         |
| AUC-ROC        | ~0.77          | ~0.86         |

*(Os valores exatos dependem da execução e do dataset gerado.)*

### Conclusões

- O **Random Forest** supera a Regressão Logística em todas as métricas, com AUC-ROC significativamente maior, indicando melhor capacidade discriminativa.
- A **Regressão Logística** serve como baseline sólido e é mais interpretável, adequada quando a explicabilidade do modelo é prioritária.
- As features mais importantes para o Random Forest são `dep_delay_min`, `carrier_delay_min` e condições climáticas como `STORM`, confirmando os insights da EDA.
- A validação cruzada mostra resultados estáveis, indicando que os modelos **não estão sofrendo de overfitting severo**.

### Limitações

- **Dependência da feature `dep_delay_min`**: na prática real, o atraso na partida pode não estar disponível no momento da previsão (antes do voo). É recomendável treinar versões dos modelos sem essa feature.
- **Dados sintéticos**: os padrões foram gerados artificialmente e podem não capturar toda a complexidade dos atrasos reais.
- **Otimização de hiperparâmetros**: não foi realizada busca exaustiva (GridSearch/RandomSearch), o que poderia melhorar os resultados.